#### Import

In [179]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

#### Data loading

In [ ]:
DATA_DIR = Path(".")
ratings = pd.read_csv(
    DATA_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    DATA_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],encoding="latin-1"
)
users = pd.read_csv(
    DATA_DIR / "users.dat",
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"]
)
print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)
print("Users shape:", users.shape)

n_users = ratings["user_id"].nunique()
n_items = ratings["movie_id"].nunique()
n_interactions = len(ratings)

print("no of sers:", n_users)
print("no of items or movies:", n_items)
print("Interactions (total number of rating):", n_interactions)

#### Checking start, end date and density

In [ ]:
ratings["datetime"] = pd.to_datetime(ratings["timestamp"],unit="s")
print("Start:", ratings["datetime"].min())
print("End:", ratings["datetime"].max())

possible_interactions = n_users * n_items

density = n_interactions / possible_interactions

print("\nDensity:", density)
print("Density %:", round(density * 100, 2))

#### User interaction

In [ ]:
user_interactions = ratings.groupby("user_id").size()

plt.figure(figsize=(8, 5))
plt.hist(user_interactions.values,bins=20)
plt.xlabel("Number of ratings per user")
plt.ylabel("Number of users")
plt.title("Distribution of interactions per user")

plt.show()


#### Item interactions and long_tail distribution of movie

In [ ]:
item_interactions = ratings.groupby("movie_id").size()
sorted_item_interactions = item_interactions.sort_values(ascending=False)
plt.figure(figsize=(10, 5))

plt.plot(
    range(len(sorted_item_interactions)),
    sorted_item_interactions.values
)

plt.xlabel("Movies sorted by popularity")
plt.ylabel("Number of ratings")
plt.title("Long-tail distribution of movie popularity")

plt.show()

#### Top 10% most active users and most rated movie

In [ ]:
top_users = user_interactions.nlargest(
    int(np.ceil(len(user_interactions) * 0.10))
)
user_share = top_users.sum() / len(ratings) * 100
top_movies = item_interactions.nlargest(
    int(np.ceil(len(item_interactions) * 0.10))
)
movie_share = top_movies.sum() / len(ratings) * 100
print(f"Top 10% of users contribute {user_share:.2f}% of all interactions")
print(f"Top 10% of movies receive {movie_share:.2f}% of all interactions")


#### Rating distribution

In [ ]:
rating_distribution = ratings["rating"].value_counts().sort_index()

plt.figure(figsize=(7, 5))

rating_distribution.plot(kind="bar")

plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.title("Rating distribution")

plt.show()

#### Rating percentage

In [ ]:
rating_percentage = (ratings["rating"].value_counts(normalize=True).sort_index()* 100)
print(rating_percentage)

ratings["month"] = ratings["datetime"].dt.to_period("M")

monthly_activity = ratings.groupby("month").size()

plt.figure(figsize=(12, 5))

plt.plot(monthly_activity.index.astype(str),
    monthly_activity.values)

plt.xlabel("Month")
plt.ylabel("Number of ratings")
plt.title("Rating activity over time")

plt.xticks(rotation=90)

plt.show()


#### Feedback treatment

In [ ]:
POSITIVE_THRESOLD = 4
ratings['relevant']= ratings['rating']>=POSITIVE_THRESOLD
ratings["relevant"].value_counts(normalize=True) * 100

#### Cuting point


In [ ]:
ratings = ratings.sort_values("timestamp").reset_index(drop=True)
train_cut = ratings["timestamp"].quantile(0.80)
val_cut = ratings["timestamp"].quantile(0.90)

train_cut_date = pd.to_datetime(train_cut, unit="s")
val_cut_date = pd.to_datetime(val_cut, unit="s")

print("Train cut:", train_cut_date)
print("Validation cut:", val_cut_date)

#### Temporal split

In [ ]:
train = ratings[
    ratings["timestamp"] <= train_cut
].copy()

validation = ratings[
    (ratings["timestamp"] > train_cut) &
    (ratings["timestamp"] <= val_cut)
].copy()

test = ratings[
    ratings["timestamp"] > val_cut
].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
total = len(ratings)

print("Train %:", round(len(train) / total * 100, 2))
print("Validation %:", round(len(validation) / total * 100, 2))
print("Test %:", round(len(test) / total * 100, 2))

print('\n Time range check')

print(
    "Train:",
    train["datetime"].min(),
    "to",
    train["datetime"].max()
)

print(
    "Validation:",
    validation["datetime"].min(),
    "to",
    validation["datetime"].max()
)

print(
    "Test:",
    test["datetime"].min(),
    "to",
    test["datetime"].max()
)

#### User and items 

In [ ]:
train_users = set(train["user_id"])
validation_users = set(validation["user_id"])
test_users = set(test["user_id"])

print(
    "Validation users unseen in train:",
    len(validation_users - train_users)
)

print(
    "Test users unseen in train:",
    len(test_users - train_users)
)

train_items = set(train["movie_id"])
validation_items = set(validation["movie_id"])
test_items = set(test["movie_id"])

print(
    "\nValidation items unseen in train:",
    len(validation_items - train_items)
)

print(
    "Test items unseen in train:",
    len(test_items - train_items)
)

#### Full catalogue

In [ ]:
catalogue = pd.Index(movies["movie_id"].drop_duplicates(),name="movie_id")
print('catalogue_size\t', len(catalogue))
catalogue

#### Most popular variant A (all popular movie)

In [ ]:
popular_all= (train.groupby('movie_id').size()
    .reindex(catalogue,fill_value=0).sort_values(ascending= False))
popular_all.head(10)
# giving 0 for non rated movie for movie train period


#### Number of rating = popularity rating, top 10

In [ ]:
top10= popular_all.head(10).rename('popularity_score').reset_index()
top10_all = top10.merge(movies[['movie_id','title']],on = 'movie_id',how='left')
top10_all

#### Positive or relevant rating only- Variant B

In [ ]:
train_positive= train[train['relevant']]

popular_positive= (train_positive.groupby('movie_id').size()
    .reindex(catalogue,fill_value=0).sort_values(ascending=False))
top10_positive = (popular_positive.head(10).rename('popularity_score')
    .reset_index().merge(movies[['movie_id','title']], on= 'movie_id', how='left'))
top10_positive


##### User specific recommendation function

In [ ]:
def recommend_popular(user_id, popularity_scores, history, n=10):

    seen_items = set(history.loc[history["user_id"] == user_id,"movie_id"])

    unseen_mask = ~popularity_scores.index.isin(seen_items)

    candidates = popularity_scores[unseen_mask] ##movie id

    return candidates.head(n)

In [ ]:
def recommendation_titles(recommendations, movies):
    return (recommendations.rename("score").reset_index().merge(movies[["movie_id", "title"]],on="movie_id",how="left"))

In [ ]:
example_user = train['user_id'].iloc[0]
first_func = recommend_popular(example_user, popular_all, train,n=10)
second_fun = recommendation_titles(first_func,movies)
second_fun

#### Making dic of movie already rated by user in training data

In [ ]:
train_seen= train.groupby('user_id').apply(set).to_dict()
#train_seen


#### Positive or relavant movie

In [ ]:
validation_truth =(validation[validation['relevant']].groupby('user_id')
                   ['movie_id'].apply(set).to_dict())

#### We delete movie seen in training and put movies which are inside catalogue

In [ ]:
catalogue_set = set(catalogue)
cleaned_truth={}
for user, items in validation_truth.items():
    seen = train_seen.get(user,set())
    unseen_items = items-seen
    valid_items = unseen_items & catalogue_set
    if valid_items:
        cleaned_truth[user] = valid_items
validation_truth = cleaned_truth
example_user = next(iter(validation_truth)) ##it gives next users

rec_all = recommend_popular(example_user,popular_all,train,n=10)
# get recommended movie id
recommended_items = set(rec_all.index)
## Get relevant movie from validation
actual_relevant = validation_truth[example_user]
## Count correctly recommended movies
hits = len(recommended_items & actual_relevant)

## Calculate precision@10
precision_at_10 = hits / 10
## Recal @10
recall_at_10 = hits / len(actual_relevant)

# Display results
print("User:", example_user)
print("Recommended movies:", recommended_items)
print("Actual relevant movies:", actual_relevant)
print("Relevant movies:", len(actual_relevant))
print("Correct recommendations:", hits)
print("Precision@10:", precision_at_10)
print("Recall@10:", recall_at_10)

In [ ]:
import numpy as np

def ranking_metrics_at_k(recommendations, relevant_items, k=10):
    recommendations = list(recommendations)[:k]
    relevant_items = set(relevant_items)
    hits = len(set(recommendations) & relevant_items)
    precision = hits / k
    recall = hits / len(relevant_items) if relevant_items else 0.0
    dcg = 0.0
    for rank, movie_id in enumerate(recommendations, start=1):
        if movie_id in relevant_items:
            dcg += 1 / np.log2(rank + 1)
    ideal_hits = min(k, len(relevant_items))
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return precision, recall, ndcg

def evaluate_popular(popularity_scores, validation_truth, train_seen, catalogue, k=10):
    precisions = []
    recalls = []
    ndcgs = []
    all_recommended = set()
    for user, relevant in validation_truth.items():
        seen = train_seen.get(user, set())
        recommendations = []
        for movie_id in popularity_scores.index:
            if movie_id not in seen:
                recommendations.append(movie_id)

            if len(recommendations) == k:
                break

        p, r, n = ranking_metrics_at_k(recommendations,relevant,k)
        
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)
        all_recommended.update(recommendations)
        
    coverage = len(all_recommended) / len(catalogue)

    return {
        "Users": len(validation_truth),
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "NDCG@10": np.mean(ndcgs),
        "Coverage": coverage}

In [ ]:
#### For all popular variant A variant B

In [ ]:
result_all = evaluate_popular(popular_all,validation_truth,train_seen,catalogue,k=10)
print(result_all)

result_positive = evaluate_popular(popular_positive,validation_truth,train_seen,catalogue,k=10)

print(result_positive)
results = pd.DataFrame([result_all, result_positive],
    index=["Most Popular - All Ratings", "Most Popular - Positive Ratings"])

results.round(4)

#### Most popular-all rating for all users

In [ ]:
all_results = []
for user, relevant in validation_truth.items():
    recommendations = recommend_popular(user,popular_all,train,n=10)
    p,r,ndcg = ranking_metrics_at_k(recommendations.index,relevant,k=10)
    all_results.append({"User": user,"Precision@10": p,"Recall@10": r,"NDCG@10": ndcg})
    
all_results = pd.DataFrame(all_results)

print("Total users:", len(all_results))
display(all_results.head(5).round(4))

##### Most popular-all rating for positive rating

In [ ]:
positive_results = []
for user, relevant in validation_truth.items():
    recommendations = recommend_popular(user,popular_positive,train,n=10)
    p,r,ndcg = ranking_metrics_at_k(recommendations.index,relevant,k=10)
    positive_results.append({"User": user,"Precision@10": p,"Recall@10": r,"NDCG@10": ndcg})
    
positive_results = pd.DataFrame(positive_results)

print("Total users:", len(positive_results))
display(positive_results.head(5).round(4))

##### Comparision

In [ ]:
print("Most Popular – All Ratings")
display(all_results[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ].mean().round(4))
print("Most Popular – Positive Ratings")
display(
    positive_results[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ].mean().round(4)
)

In [ ]:
# Select only positive training interactions
positive_train = train.loc[train['relevant'], ["user_id", "movie_id"]].drop_duplicates()
user_ids = pd.Index(train["user_id"].unique(), name="user_id")
movie_ids = catalogue
# Convert actual IDs to matrix positions
user_positions = user_ids.get_indexer(positive_train["user_id"])
movie_positions = movie_ids.get_indexer(positive_train["movie_id"])
# Create sparse user-item matrix.. 
# matrix has empty cell and sparse matrix put nonzero and store position
X = csr_matrix((
        np.ones(len(positive_train), dtype=np.float32),
        (user_positions, movie_positions)##rows,columns
    ),
    shape=(len(user_ids), len(movie_ids)))

print("Matrix shape:", X.shape)
print("Positive interactions:", X.nnz)


#### Similarity

In [ ]:
#  Find Toy Story's matrix column
toy_index = movie_ids.get_loc(1)
# Calculate Toy Story similarity with every movie
toy_similarity = cosine_similarity(X.T[toy_index],X.T).ravel()
similarity_scores = pd.Series(toy_similarity,index=movie_ids,name="similarity")
similarity_scores = similarity_scores.drop(index=1)
top_similar = (similarity_scores
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .merge(movies[["movie_id", "title"]],on="movie_id",how="left"))

top_similar

In [ ]:
# Similarity between every pair of movies
item_similarity = cosine_similarity(X.T).astype(np.float32)

# Remove self-similarity
np.fill_diagonal(item_similarity, 0)

K = 20
neighbors = np.argsort(-item_similarity,axis=1,kind="stable")[:, :K]

neighbor_weights = np.take_along_axis(item_similarity,neighbors,axis=1)
# User ID to matrix row position
user_to_row = {user_id: i for i, user_id in enumerate(user_ids)}


# Movie popularity used to break equal scores
popularity_tiebreak = popular_all.reindex(movie_ids, fill_value=0).to_numpy()

def recommend_item_knn(user_id, n=10):

    # Positive movie history of this user
    liked = np.zeros(len(movie_ids), dtype=np.float32)

    if user_id in user_to_row:
        liked = X[user_to_row[user_id]].toarray().ravel()

    # Score every candidate movie using its top-K neighbors
    scores = (neighbor_weights * liked[neighbors]).sum(axis=1)

    # Exclude all movies already rated in training
    seen = train_seen.get(user_id, set())

    seen_positions = movie_ids.get_indexer(list(seen))
    seen_positions = seen_positions[seen_positions >= 0]

    scores[seen_positions] = -np.inf

    # Rank by score, then popularity, then movie ID
    ranked_positions = np.lexsort((movie_ids.to_numpy(),-popularity_tiebreak,-scores))

    top_positions = ranked_positions[:n]

    return pd.Series(
        scores[top_positions],
        index=movie_ids[top_positions],
        name="knn_score")

In [ ]:
example_user = next(iter(validation_truth))

knn_recs = recommend_item_knn(example_user,n=10)

print(knn_recs)
knn_titles = (knn_recs
    .rename("score")
    .reset_index()
    .merge(
        movies[["movie_id", "title"]],
        on="movie_id",
        how="left"))

knn_titles

relevant = validation_truth[example_user]

p, r, ndcg = ranking_metrics_at_k(knn_recs.index,relevant,k=10)

print("User:", example_user)
print("Precision@10:", round(p, 4))
print("Recall@10:", round(r, 4))
print("NDCG@10:", round(ndcg, 4))

#### Jaccard

In [ ]:
# Number of positive interactions for each movie

item_counts = np.asarray(X.sum(axis=0)).ravel().astype(np.float32)
item_counts
# Number of users who liked both movies
intersection = (X.T @ X).toarray().astype(np.float32)
# Number of users who liked at least one
union = (item_counts[:, None]+ item_counts[None, :]- intersection)
# Calculate Jaccard
jaccard_similarity = np.divide(intersection,union,out=np.zeros_like(intersection),where=union > 0)

# Exclude self similarity
np.fill_diagonal(jaccard_similarity, 0)

print("Jaccard shape:", jaccard_similarity.shape)

#### KNN

In [ ]:
def build_knn_weights(similarity, k):
    n_items = similarity.shape[0]
    if not 1 <= k < n_items:
        raise ValueError("Invalid k")
    # Copy similarity matrix
    sim = similarity.copy()
    # Exclude each movie from its own neighbors
    np.fill_diagonal(sim, -np.inf)

    # Select top-k similar movie indices
    neighbors = np.argsort(-sim, axis=1, kind="stable")[:, :k]

    # Get actual similarity values
    weights = np.take_along_axis(similarity, neighbors, axis=1)

    # Create sparse neighbor-weight matrix
    W = csr_matrix((weights.ravel(),
            (np.repeat(np.arange(n_items), k),
                neighbors.ravel()
            )),shape=(n_items, n_items))

    W.eliminate_zeros()

    return W.T.tocsr()

In [ ]:
# User ID → matrix row
user_to_row = {uid: i for i, uid in enumerate(user_ids)}

# Popularity used for deterministic tie-breaking
popularity_tiebreak = popular_all.reindex(movie_ids, fill_value=0).to_numpy()


def recommend_knn(user_id, weights_T, n=10):
    scores = np.zeros(len(movie_ids), dtype=np.float32)

    # Calculate item-kNN scores
    if user_id in user_to_row:
        scores = (X[user_to_row[user_id]] @ weights_T).toarray().ravel()

    # Remove previously rated movies
    seen = train_seen.get(user_id, set())

    seen_positions = movie_ids.get_indexer(list(seen))

    seen_positions = seen_positions[seen_positions >= 0]

    scores[seen_positions] = -np.inf

    # Sort by score, popularity, then movie ID
    ranked = np.lexsort((movie_ids.to_numpy(),-popularity_tiebreak,-scores))

    # Select top-n valid candidates
    top = ranked[np.isfinite(scores[ranked])][:n]

    return pd.Series(scores[top],index=movie_ids[top],name="knn_score")

In [ ]:
def evaluate_knn(weights_T):

    precisions = []
    recalls = []
    ndcgs = []

    all_recommended = set()

    for user, relevant in validation_truth.items():
        recommendations = recommend_knn(user, weights_T, n=10)

        p, r, ndcg = ranking_metrics_at_k(recommendations.index,relevant,k=10)

        precisions.append(p)
        recalls.append(r)
        ndcgs.append(ndcg)
        all_recommended.update(recommendations.index)

    return {
        "Users": len(validation_truth),
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "NDCG@10": np.mean(ndcgs),
        "Coverage": len(all_recommended) / len(catalogue)
    }

#### Comparing results

In [ ]:
experiments = []

similarities = {"Cosine": item_similarity,"Jaccard": jaccard_similarity}

k_values = [10, 20, 50]

for sim_name, sim_matrix in similarities.items():

    for k in k_values:

        print("Evaluating:", sim_name, "k =", k)

        weights_T = build_knn_weights(sim_matrix, k)

        metrics = evaluate_knn(weights_T)

        experiments.append({"Similarity": sim_name,"k": k,**metrics})

knn_results = pd.DataFrame(experiments)

knn_results = knn_results.sort_values(["NDCG@10", "Precision@10"],ascending=False).reset_index(drop=True)

display(knn_results.round(4))


In [ ]:
knn_comparison = knn_results.copy()

knn_comparison["Model"] = ("kNN "+ knn_comparison["Similarity"]+ " k="
    + knn_comparison["k"].astype(str))
# Add model names to popularity results
popular_comparison = results.reset_index().rename(columns={"index": "Model"})

comparison = pd.concat([popular_comparison, knn_comparison],ignore_index=True)
display(comparison[["Model","Precision@10","Recall@10", "NDCG@10", "Coverage"]].round(4))